# Loan Amount Prediction — Exploratory Data Analysis

Before writing a single line of model code, I want to understand the data well enough to have opinions about it. The specific questions I'm trying to answer:

- What does the loan amount distribution actually look like? Is it skewed? Are there clusters?
- Which features correlate with how much someone borrows?
- Where is the data messy (missing values, outliers)?
- What will hurt a neural network if left unhandled?

The findings here drive every decision in `02_preprocessing.ipynb`.

## The Problem

Given a borrower's profile — income, employment, home ownership, credit history, loan intent and grade — predict the **loan amount** they are likely to be approved for.

This is a regression problem. The target is a continuous dollar value ranging from $500 to $35,000.

### Why this is harder than it looks

Loan amount is primarily what the **borrower decided to request**, not something the lender derives from the credit profile. A borrower with $80k income might request $5,000 for a small personal expense, or $30,000 for a home renovation. Their income tells us something about capacity, but not about intent or need. That fundamental disconnect — between what the features measure and what we're trying to predict — is why R² ends up modest even with a well-trained model. That's not a failure. It's an honest reflection of the problem.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
os.makedirs('../outputs/figures', exist_ok=True)

df = pd.read_csv('../data/raw/credit_risk.csv')
print(f'Loaded {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')

## Data Shape & Types

In [ ]:
print('=== Dtypes ===')
print(df.dtypes)
print('\n=== Summary stats ===')
df.describe().T.round(2)

## Missing Values

Two columns have missing data:
- **`loan_int_rate`**: 3,116 missing (9.6%) — the most important numeric predictor, can't drop it
- **`person_emp_length`**: 895 missing (2.7%) — moderate, safe to impute

The original code used `dropna(axis=1)` which deletes entire **columns** with any missing values. That silently deleted both of these. I use imputation instead — grade-grouped median for interest rate, global median for employment length.

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
miss_df = pd.DataFrame({'Count': missing, 'Pct': missing_pct}).query('Count > 0')

fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.barh(miss_df.index, miss_df['Pct'],
               color=['#C62828' if p>5 else '#FF9800' for p in miss_df['Pct']])
for bar, (col, row) in zip(bars, miss_df.iterrows()):
    ax.text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2,
            f"{row['Count']:,} rows ({row['Pct']}%)", va='center', fontsize=10)
ax.set_xlabel('Missing %')
ax.set_title('loan_int_rate is too predictive to drop — impute it instead', fontweight='bold')
ax.set_xlim(0, 15)
plt.tight_layout()
plt.savefig('../outputs/figures/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

## Target Distribution: `loan_amnt`

> **All values are in USD.** The dataset is from a US lending platform.
> Valid range: **$500 – $35,000**. Mean: $9,589. Median: $8,000.
>
> If you enter values outside this range into the model, predictions become meaningless.
> The demo in the original Flutter video used a loan input of 500,000 — that's 14× above
> the maximum value the model was ever trained on. The output of 550,034 was extrapolation noise,
> not a real prediction. Valid inputs must stay within the training distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['loan_amnt'], bins=60, color='#1565C0', edgecolor='white', alpha=0.85)
axes[0].axvline(df['loan_amnt'].mean(),   color='red',    lw=2, linestyle='--', label=f'Mean = ${df["loan_amnt"].mean():,.0f}')
axes[0].axvline(df['loan_amnt'].median(), color='orange', lw=2, linestyle='--', label=f'Median = ${df["loan_amnt"].median():,.0f}')
axes[0].set_title('Loan amounts cluster below $15k — heavily right-skewed', fontweight='bold')
axes[0].set_xlabel('Loan Amount (USD)'); axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[0].legend()

loan_intent_avg = df.groupby('loan_intent')['loan_amnt'].mean().sort_values(ascending=False)
axes[1].bar(loan_intent_avg.index, loan_intent_avg.values / 1000,
            color='#1565C0', edgecolor='white')
axes[1].set_title('Loan intent — debt consolidation & home improvement = larger loans', fontweight='bold')
axes[1].set_xlabel('Loan Intent'); axes[1].set_ylabel('Avg Loan Amount ($k)')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('../outputs/figures/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Min:    ${df["loan_amnt"].min():,}')
print(f'Max:    ${df["loan_amnt"].max():,}')
print(f'Mean:   ${df["loan_amnt"].mean():,.0f}')
print(f'Median: ${df["loan_amnt"].median():,.0f}')

## Feature Distributions vs Loan Amount

In [ ]:
numeric_cols = ['person_age', 'person_income', 'person_emp_length', 'loan_int_rate', 'cb_person_cred_hist_length']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].scatter(df[col], df['loan_amnt']/1000, alpha=0.15, s=8, color='#1565C0')
    axes[i].set_xlabel(col); axes[i].set_ylabel('Loan Amount ($k)')
    axes[i].set_title(col, fontweight='bold')
    axes[i].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:.0f}k'))

axes[-1].axis('off')
plt.suptitle('Weak scatter relationships — no single feature predicts loan amount cleanly', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/feature_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()

## Correlation Matrix

This is the honest picture. The correlations with `loan_amnt` are low across the board. The highest is `person_income` at around 0.34. That means income explains about 11% of the variance in loan amount on its own — not nothing, but not strong either.

This is the core challenge: the features in this dataset were designed to assess credit *risk*, not to capture why someone needs a specific dollar amount.

In [ ]:
num_for_corr = ['person_age','person_income','person_emp_length','loan_amnt','loan_int_rate','cb_person_cred_hist_length']
corr = df[num_for_corr].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5, annot_kws={'size': 11})
ax.set_title('Low correlations with loan_amnt — borrowers choose amounts independently of credit profile',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('Correlations with loan_amnt:')
for f, v in corr['loan_amnt'].drop('loan_amnt').sort_values(key=abs, ascending=False).items():
    print(f'  {f:<30} {v:+.3f}')

## Categorical Feature Breakdown

In [ ]:
cat_cols = ['loan_grade', 'person_home_ownership', 'loan_intent', 'cb_person_default_on_file']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    avg = df.groupby(col)['loan_amnt'].mean().sort_values(ascending=False)
    axes[i].bar(avg.index, avg.values/1000, color='#1565C0', edgecolor='white')
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel('Avg Loan Amount ($k)')
    axes[i].tick_params(axis='x', rotation=15)

plt.suptitle('Categorical averages — modest differences, no single category dominates', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/categorical_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()

## Key EDA Findings

**1. The target is USD, range $500–$35,000, with heavy right skew.**
Most loans cluster under $15k. Round numbers (5,000, 10,000, 15,000, 20,000) dominate — people tend to ask for round amounts. Any demo or inference must use values in this range.

**2. Income is the strongest individual predictor — but only weakly.**
Correlation of ~0.34 with loan_amnt. Higher earners tend to borrow more, but the relationship has enormous scatter. A $50k earner might request $2,000 or $30,000.

**3. Loan intent adds modest signal.**
Debt consolidation and home improvement loans tend to be larger. Medical and education loans are more variable. Intent captures *why* someone is borrowing, which helps slightly with *how much*.

**4. The two dropped features are correct to drop.**
`loan_percent_income` = loan_amnt / person_income — direct leakage of the target.
`loan_status` is determined after the loan is issued — not available at prediction time.

**5. R² will be modest. That's expected.**
The fundamental issue: loan amount is borrower-chosen, not a function of their credit profile. The model will capture real signal from income, grade, and intent — but a significant portion of variance is simply personal choice, which no model can predict from these features.